# 第8回: ファイル操作・コマンド実行・Web検索

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cosmac-dev/ai-agent-seminar/blob/main/session08/session08_tools.ipynb)

AIエージェントが実務でよく使うツールを実装する。


---
## 0. 環境準備


In [ ]:
# @markdown 実行環境フラグ: Google Colab で実行する場合は True にする
IS_COLAB = False # @param {type:"boolean"}

In [ ]:
# @title 依存パッケージのインストール
%pip install -qU langchain langchain-community langchain-experimental langchain-openai langchain-tavily langgraph python-dotenv

In [ ]:
import os
import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY を入力する: ")

if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("TAVILY_API_KEY を入力する: ")

print("OpenAI APIキー設定完了" if os.environ.get("OPENAI_API_KEY") else "OpenAI APIキー未設定")
print("Tavily APIキー設定完了" if os.environ.get("TAVILY_API_KEY") else "Tavily APIキー未設定")


In [ ]:
# @title importと初期設定
import getpass
import os
from pathlib import Path


WORKSPACE_DIR = Path("agent_workspace").resolve()
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)
print("Tool用ワークスペース:", WORKSPACE_DIR)


---
## 1. Web検索

| 機能 | 関数 | 概要 |
|---|---|---|
| Search API / Model built-in search | `web_search` | 最新情報や公開情報を検索する |
| Web fetcher / Extract API | `fetch_url` | URLを指定してページ本文を取得する |


In [ ]:
# @title web_search
from langchain_tavily import TavilySearch

web_search = TavilySearch(
    name="web_search",
    max_results=3,
    search_depth="basic",
    include_answer=False,
    include_raw_content=False,
)
print(f"{web_search.name}: {web_search.description}")


In [ ]:
# @title web_search の動作確認
web_search_result = web_search.invoke(
    {"query": "LangChain Python tools official documentation"}
)
web_search_result


In [ ]:
# @title fetch_url
from langchain_tavily import TavilyExtract

fetch_url = TavilyExtract(
    name="fetch_url",
    extract_depth="basic",  # "basic": 本文テキストのみ / "advanced": 表や埋め込みも含める
    include_images=False,  # 抽出結果に画像URLの一覧を含めるか
    format="markdown",  # 本文の形式。"markdown" または "text"
    chunks_per_source=1,  # 1つのソースから抽出するチャンク数
)
print(f"{fetch_url.name}: {fetch_url.description}")


In [ ]:
# @title fetch_url の動作確認
fetch_url_result = fetch_url.invoke(
    {"urls": ["https://docs.langchain.com/oss/python/langchain/tools"]}
)
fetch_url_result


---
## 2. ファイル操作

| 機能 | 関数 | 概要 |
|---|---|---|
| ファイル書き込み | `write_file` | ファイルに内容を書き込む |
| ファイル読み取り | `read_file` | ファイルの内容を読み取る |
| ファイルコピー | `copy_file` | ファイルをコピーする |
| ファイル移動 | `move_file` | ファイルを移動またはリネームする |
| ファイル削除 | `file_delete` | ファイルを削除する |
| ディレクトリ一覧 | `list_directory` | ディレクトリ内のファイル一覧を取得する |
| ファイル検索 | `file_search` | ワークスペース内のファイルを検索する |

In [ ]:
# @title write_file
from langchain_community.tools.file_management import WriteFileTool

write_file = WriteFileTool(root_dir=str(WORKSPACE_DIR))
print(f"{write_file.name}: {write_file.description}")


In [ ]:
# @title write_file の動作確認
print(
    write_file.invoke(
        {
            "file_path": "notes.txt",
            "text": "web_searchで調査し、write_fileで結果を保存する。\n",
        }
    )
)


In [ ]:
# @title read_file
from langchain_community.tools.file_management import ReadFileTool

read_file = ReadFileTool(root_dir=str(WORKSPACE_DIR))
print(f"{read_file.name}: {read_file.description}")


In [ ]:
# @title read_file の動作確認
print(read_file.invoke({"file_path": "notes.txt"}))


In [ ]:
# @title copy_file
from langchain_community.tools.file_management import CopyFileTool

copy_file = CopyFileTool(root_dir=str(WORKSPACE_DIR))
print(f"{copy_file.name}: {copy_file.description}")


In [ ]:
# @title copy_file の動作確認
print(
    copy_file.invoke(
        {"source_path": "notes.txt", "destination_path": "notes_copy.txt"}
    )
)


In [ ]:
# @title move_file
from langchain_community.tools.file_management import MoveFileTool

move_file = MoveFileTool(root_dir=str(WORKSPACE_DIR))
print(f"{move_file.name}: {move_file.description}")


In [ ]:
# @title move_file の動作確認
print(
    move_file.invoke(
        {"source_path": "notes_copy.txt", "destination_path": "notes_moved.txt"}
    )
)


In [ ]:
# @title file_delete
from langchain_community.tools.file_management import DeleteFileTool

file_delete = DeleteFileTool(root_dir=str(WORKSPACE_DIR))
print(file_delete.name, file_delete.description)


In [ ]:
# @title file_delete の動作確認
print(file_delete.invoke({"file_path": "notes_moved.txt"}))


In [ ]:
# @title list_directory
from langchain_community.tools.file_management import ListDirectoryTool

list_directory = ListDirectoryTool(root_dir=str(WORKSPACE_DIR))
print(f"{list_directory.name}: {list_directory.description}")


In [ ]:
# @title list_directory の動作確認
print(list_directory.invoke({"dir_path": "."}))


In [ ]:
# @title file_search
from langchain_community.tools.file_management import FileSearchTool

file_search = FileSearchTool(root_dir=str(WORKSPACE_DIR))
print(f"{file_search.name}: {file_search.description}")


In [ ]:
# @title file_search の動作確認
print(file_search.invoke({"pattern": "*.txt", "dir_path": "."}))


---
## 3. HTTP

| 機能 | 関数 | 概要 |
|---|---|---|
| GET | `requests_get` | URLを指定してリソースを取得する |
| POST | `requests_post` | URLを指定してデータを送信する |
| PATCH | `requests_patch` | URLを指定してリソースを部分更新する |
| PUT | `requests_put` | URLを指定してリソースを更新する |
| DELETE | `requests_delete` | URLを指定してリソースを削除する |


In [ ]:
# @title requests_get
from langchain_community.tools.requests.tool import RequestsGetTool
from langchain_community.utilities.requests import TextRequestsWrapper

requests_wrapper = TextRequestsWrapper(
    headers={"User-Agent": "ai-agent-seminar-session08/1.0"}
)
requests_get = RequestsGetTool(
    requests_wrapper=requests_wrapper,
    allow_dangerous_requests=True,
)
print(f"{requests_get.name}: {requests_get.description}")


In [ ]:
# @title requests_get の動作確認
requests_get_result = requests_get.invoke(
    {"url": "https://jsonplaceholder.typicode.com/todos/1"}
)
print(requests_get_result)


In [ ]:
# @title requests_post
from langchain_community.tools.requests.tool import RequestsPostTool

requests_post = RequestsPostTool(
    requests_wrapper=requests_wrapper,
    allow_dangerous_requests=True,
)
print(f"{requests_post.name}: {requests_post.description}")


In [ ]:
# @title requests_post の動作確認
import json

print(
    requests_post.invoke(
        {
            "text": json.dumps(
                {
                    "url": "https://jsonplaceholder.typicode.com/posts",
                    "data": {
                        "title": "session08",
                        "body": "HTTP POST demo",
                        "userId": 1,
                    },
                }
            )
        }
    )
)


In [ ]:
# @title requests_patch
from langchain_community.tools.requests.tool import RequestsPatchTool

requests_patch = RequestsPatchTool(
    requests_wrapper=requests_wrapper,
    allow_dangerous_requests=True,
)
print(f"{requests_patch.name}: {requests_patch.description}")


In [ ]:
# @title requests_patch の動作確認
print(
    requests_patch.invoke(
        {
            "text": json.dumps(
                {
                    "url": "https://jsonplaceholder.typicode.com/posts/1",
                    "data": {"title": "session08 patched"},
                }
            )
        }
    )
)


In [ ]:
# @title requests_put
from langchain_community.tools.requests.tool import RequestsPutTool

requests_put = RequestsPutTool(
    requests_wrapper=requests_wrapper,
    allow_dangerous_requests=True,
)
print(f"{requests_put.name}: {requests_put.description}")


In [ ]:
# @title requests_put の動作確認
print(
    requests_put.invoke(
        {
            "text": json.dumps(
                {
                    "url": "https://jsonplaceholder.typicode.com/posts/1",
                    "data": {
                        "id": 1,
                        "title": "session08 put",
                        "body": "HTTP PUT demo",
                        "userId": 1,
                    },
                }
            )
        }
    )
)


In [ ]:
# @title requests_delete
from langchain_community.tools.requests.tool import RequestsDeleteTool

requests_delete = RequestsDeleteTool(
    requests_wrapper=requests_wrapper,
    allow_dangerous_requests=True,
)
print(f"{requests_delete.name}: {requests_delete.description}")


In [ ]:
# @title requests_delete の動作確認
print(
    requests_delete.invoke(
        {"url": "https://jsonplaceholder.typicode.com/posts/1"}
    )
)


---
## 4. コマンド・コード実行

| 機能 | 関数 | 概要 |
|---|---|---|
| コマンド実行 | `run_command` | OSのシェルコマンドを実行する |
| コード実行 | `python_repl` | Pythonコードを実行する |



In [ ]:
# @title run_command
from langchain_community.tools.shell.tool import ShellTool

run_command = ShellTool(
    name="run_command",
    ask_human_input=False,
)
print(f"{run_command.name}: {run_command.description}")


In [ ]:
# @title run_command の動作確認
cwd_command = "cd" if os.name == "nt" else "pwd"
print(run_command.invoke({"commands": cwd_command}))


In [ ]:
# @title python_repl
from langchain.tools import tool
from langchain_experimental.utilities import PythonREPL

_python_repl = PythonREPL()

@tool
def python_repl(code: str) -> str:
    """Pythonコードを実行する。結果として確認したい値はprintで出力すること。"""
    return _python_repl.run(code)

print(f"{python_repl.name}: {python_repl.description}")


In [ ]:
# @title python_repl の動作確認
print(
    python_repl.invoke(
        {"code": "values = [3, 5, 8, 13]\nprint(sum(values) / len(values))"}
    )
)


---
## 5. エージェントにツールを追加

[![](https://mermaid.ink/img/pako:eNqNks9uhCAQxl9lM8dGjawKyKGX9toXaG02RKhrCrJB3O7W-O5VGuKuvZQLzDe_-fg3I9RGSGDQO-7kc8sby3V83le26nbzeHt438Xx404ZLg5aamOvIXUjeaTmSh307KYCsSor4IxR_R3glf85yIuzvHabg9yrHjxz1Yr5QhtyI3v0y7Z_uFvNQ_MzhFyYIYLGtgKYs4OMQEur-RLCuOQrcEepZQVsXgpuPyuoummuOfHu1RgdyqwZmmMIhpNYPwHYB1f9gshOSPtkhs4BK5G3ADbCBRjCJKFFnqMCI4rTAmcRXIFlKCGU4CxNU4JRhskUwbffNE1oWWQUlZSkZU5JuY9AitYZ-_LbBr4bph9o96vK?type=png)](https://mermaid.live/edit#pako:eNqNks9uhCAQxl_FzLFRg6sCcuilvfYFWhpDhLimIhsWt7s1vntdNtSuvZQLzPf9Zvg3EzRGKmBwdMKp5060VujktOOWD9Ey3h7eoyR5jHojZK2VNvYSrF-SRxrR97VeqvWBWJUVcMb0xzvAK_-roM7OisZtDnKvevAk-k4uF9qQG9mjn7b7w92i2lseWp4heGGGGFrbSWDOjioGrawW1xCmq8_B7ZVWHNiylMJ-cODDvOQcxPBqjA5p1oztPgTjQa6f8EOoQSr7ZMbBAaPEVwA2wRlYhklKy6LISpxRjEqcx3ABlmcpoQTnCCGCsxyTOYYvvydKaVXmNKsoQVVBSbWLQcnOGfty6wLfDPM3vDirhA)

In [ ]:
# @title Tool定義
tools = [
    list_directory,
    read_file,
    write_file,
    copy_file,
    move_file,
    file_search,
    file_delete,
    web_search,
    fetch_url,
    requests_get,
    requests_post,
    requests_patch,
    requests_put,
    requests_delete,
    run_command,
    python_repl,
]

print("用意したツール:", [t.name for t in tools])


In [ ]:
# @title 状態
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages

# TypedDict は、決まったキーと値の型を持つ辞書を表す型
class AgentState(TypedDict):
    # 会話履歴
    # Annotated は、基本の型に追加情報を添えた型
    # add_messages は、既存のメッセージ列と新しいメッセージ列を結合する関数
    messages: Annotated[list, add_messages]

    # 長期記憶から読み込んだ情報
    memories: list[str]

    # 今回の会話から抽出された記憶候補
    memory_candidates: list[str]

    # 保存してよいと判定された記憶
    approved_memories: list[str]

In [ ]:
# @title load_memory
from dataclasses import dataclass
from uuid import uuid4

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langgraph.runtime import Runtime
from langchain_openai import ChatOpenAI

@dataclass
class Context:
    user_id: str

def load_memory(state: AgentState, runtime: Runtime[Context]) -> AgentState:
    """
    長期記憶をStoreから読む。
    現在のユーザー入力に意味的に関連する記憶だけをセマンティック検索で取得する。
    """
    user_id = runtime.context.user_id # 実行時コンテキストはグラフの実行時に渡される
    namespace = ("memories", user_id) # 保存先の名前空間 memories/{user_id} を想定

    # 直近のユーザー発話を検索クエリにする
    query = ""
    for msg in reversed(state["messages"]):
        if isinstance(msg, HumanMessage) and msg.content:
            query = str(msg.content)
            break

    # プロンプトに載せる関連記憶の最大件数
    TOP_K = 5

    if query:
        try:
            # Storeに埋め込みindexがあれば関連度順に取得される
            items = runtime.store.search(namespace, query=query, limit=TOP_K)
        except Exception:
            # index未設定などで検索できない場合は全件取得にフォールバック
            items = runtime.store.search(namespace)
    else:
        # クエリが無いターンでは検索せず全件（もしくは無し）を返す
        items = runtime.store.search(namespace)

    # "memories"フィールドだけ差分更新（上書き）
    return {
        "memories": [item.value["text"] for item in items],
    }

In [ ]:
# @title call_model
# gpt-5.6-* は /v1/chat/completions で function tools と reasoning_effort を併用できない。
# bind_tools 利用時は reasoning_effort="none"（代替: use_responses_api=True）。
model_name = "gpt-5.4-mini" # @param ["gpt-5.4-mini", "gpt-5.5", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-5.6-luna", "gpt-5.4-nano"]
use_responses_api = False # @param {type:"boolean"}
model = ChatOpenAI(
    model=model_name,
    temperature=0,
    use_responses_api=use_responses_api,
)

model_with_tools = model.bind_tools(tools)

def call_model(state: AgentState) -> AgentState:
    """
    通常のReAct用LLM node。
    ここでは長期記憶をプロンプトに差し込む。
    """

    memory_text = "\n".join(f"- {m}" for m in state["memories"])

    system = SystemMessage(
        content=f"""
あなたはAIエージェントです。
必要な場合は複数のToolを順に呼び出してください。

参考になる長期記憶:
{memory_text}
"""
    )

    response = model_with_tools.invoke(
        [system] + state["messages"]
    )

    # "messages"フィールドだけ差分更新（追記）
    return {
        "messages": [response],
    }



In [ ]:
# @title extract_memory
from pydantic import BaseModel, Field

class MemoryItem(BaseModel):
    text: str = Field(description="将来の会話でも役立つユーザー情報")

class MemoryExtraction(BaseModel):
    memories: list[MemoryItem] = Field(default_factory=list)

memory_extractor = model.with_structured_output(
    MemoryExtraction,
    method="json_schema",
)

def extract_memory(state: AgentState) -> dict:
    transcript = []
    for message in state["messages"]:
        if isinstance(message, HumanMessage):
            transcript.append(f"ユーザー: {message.content}")
        elif message.type == "ai" and isinstance(message.content, str):
            transcript.append(f"アシスタント: {message.content}")

    prompt = (
        "会話から、別の会話でも有用なユーザーの安定した好み、プロフィール、"
        "目標、制約、決定事項だけを抽出してください。"
        "一時的な依頼、Toolの入出力、今回だけのファイル名は除外してください。"
        "該当しなければ空のリストを返してください。\n\n"
        + "\n".join(transcript)
    )
    try:
        result = memory_extractor.invoke(prompt)
        candidates = [item.text for item in result.memories]
    except Exception as error:
        print("記憶候補の抽出をスキップしました:", error)
        candidates = []
    return {"memory_candidates": candidates}


In [ ]:
# @title validate_memory
import re

SENSITIVE_PATTERNS = [
    re.compile(r"(?:api[_ -]?key|password|secret|token)\s*[:=]", re.IGNORECASE),
    re.compile(r"(?:sk|ghp|github_pat)_[A-Za-z0-9_-]{12,}"),
    re.compile(r"\b(?:\d[ -]*?){13,19}\b"),
    re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+"),
    re.compile(r"\b\d{2,4}-\d{2,4}-\d{3,4}\b"),
]

def _normalize_memory(text: str) -> str:
    return " ".join(text.strip().split())

def validate_memory(state: AgentState) -> dict:
    known = {_normalize_memory(item).casefold() for item in state.get("memories", [])}
    approved = []
    for candidate in state.get("memory_candidates", []):
        normalized = _normalize_memory(candidate)
        key = normalized.casefold()
        if not normalized or key in known:
            continue
        if any(pattern.search(normalized) for pattern in SENSITIVE_PATTERNS):
            continue
        approved.append(normalized)
        known.add(key)
    return {"approved_memories": approved}


In [ ]:
# @title write_memory
from uuid import uuid4

def write_memory(state: AgentState, runtime: Runtime[Context]) -> dict:
    namespace = ("memories", runtime.context.user_id)
    for memory in state.get("approved_memories", []):
        runtime.store.put(
            namespace,
            str(uuid4()),
            {"text": memory, "source": "conversation"},
        )
    return {"memory_candidates": [], "approved_memories": []}


In [ ]:
# @title グラフ定義
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import OpenAIEmbeddings
from langgraph.store.memory import InMemoryStore
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition


embedding_model = "text-embedding-3-small" # @param ["text-embedding-3-small", "text-embedding-3-large", "text-embedding-ada-002"]
ltm_store = InMemoryStore(
    index={
        "embed": OpenAIEmbeddings(
            model=embedding_model
            ),
        "dims": 1536,
        "fields": ["text"],
    }
)

stm_checkpointer = InMemorySaver()

builder = StateGraph(AgentState)

builder.add_node(load_memory)
builder.add_node(call_model)
builder.add_node("tools", ToolNode(tools))
builder.add_node(extract_memory)
builder.add_node(validate_memory)
builder.add_node(write_memory)

builder.add_edge(START, "load_memory")
builder.add_edge("load_memory", "call_model")

builder.add_conditional_edges(
    "call_model",
    tools_condition,
    {
        "tools": "tools",
        END: "extract_memory", # ENDの判定が出たらextract_memoryにルーティング
    },
)

builder.add_edge("tools", "call_model")

builder.add_edge("extract_memory", "validate_memory")
builder.add_edge("validate_memory", "write_memory")
builder.add_edge("write_memory", END)

# Store（長期記憶）とCheckpointer（短期記憶）を渡してグラフをコンパイル
graph_with_practical_tools = builder.compile(store=ltm_store, checkpointer=stm_checkpointer)


In [ ]:
# @title 共通設定
context = Context(user_id="session08-user")
config = {
    "configurable": {"thread_id": "session08"}
}

def run(message: str, config: dict) -> dict:
    for chunk in graph_with_practical_tools.stream(
        {"messages": [HumanMessage(content=message)]},
        config=config,
        context=context,
        stream_mode="updates",
    ):
        for update in chunk.values():
            for item in update.get("messages", []):
                item.pretty_print()
    return graph_with_practical_tools.get_state(config).values


In [ ]:
# @title ファイルを作成する
demo1_result = run(
    (
        "tool_notes.mdに今回使えるToolの一覧を書いてください。"
    ),
    config,
)


In [ ]:
# @title ファイルを読む
demo2_result = run(
    (
        "さっき作ったファイルを読み、"
        "掲載したTool名を箇条書きで教えてください。"
    ),
    config,
)


In [ ]:
# @title Web検索
demo3_result = run(
    (
        "LangChainのToolに関する概要のみを公式ページから取得し、"
        "その内容をtool_research.mdへ保存してください。"
    ),
    config,
)


In [ ]:
# @title コマンドとPython実行
demo4_result = run(
    (
        "保存したファイルのサイズと文字数を確認して教えてください。"
        f"対象ディレクトリは {WORKSPACE_DIR} です。"
    ),
    config,
)


---
## 6. 実務的なエージェント: `practical_agent`


In [ ]:
if IS_COLAB:
    !git clone https://github.com/cosmac-dev/ai-agent-seminar.git
    %cd ai-agent-seminar/session08
    # セッションの再起動が要求されたら再起動する
    # 再起動後はすべてのインスタンス変数、環境変数は初期化されるので`IS_COLAB`, `OPENAI_API_KEY`, `TAVILY_API_KEY`を再設定する
    # パッケージは再起動後も残っているので、このセルの再実行は不要

%pip install -e ".[server]"
!echo "OPENAI_API_KEY=$OPENAI_API_KEY" > .env
!echo "TAVILY_API_KEY=$TAVILY_API_KEY" >> .env

In [ ]:
if IS_COLAB:
    %cd /content/ai-agent-seminar/session08
    !langgraph dev --tunnel
else:
    !langgraph dev